# 🔬✅📊 Multi-Agent Research, Validation & Analysis System with LangGraph

This notebook demonstrates a **collaborative multi-agent system** using LangGraph where a **Researcher Agent**, **Validator Agent**, and **Analyst Agent** work together iteratively to answer complex technical questions with verified, evidence-backed recommendations.

## Workflow Overview

```
┌──────────────────────────────────────────────────────────────────────────────┐
│            MULTI-AGENT RESEARCH, VALIDATION & ANALYSIS SYSTEM               │
├──────────────────────────────────────────────────────────────────────────────┤
│                                                                              │
│  ┌─────────┐    ┌──────────────────┐    ┌──────────────────┐                │
│  │  User   │───►│ RESEARCHER AGENT │───►│ VALIDATOR AGENT  │                │
│  │  Query  │    │                  │    │  (First Pass)    │                │
│  └─────────┘    │ • Search docs    │    │                  │                │
│                 │ • Find papers    │    │ • Source check   │                │
│                 │ • Get benchmarks │    │ • Contradictions │                │
│                 │ • Examples       │    │ • Flag gaps      │                │
│                 └────────┬─────────┘    └────────┬─────────┘                │
│                          │                       │                          │
│                          │◄──────────────────────┘                          │
│                          │  (Refine if gaps found)                          │
│                          ▼                                                  │
│                 ┌──────────────────┐    ┌──────────────────┐                │
│                 │  ANALYST AGENT   │───►│ VALIDATOR AGENT  │                │
│                 │                  │    │  (Second Pass)   │                │
│                 │ • Compare options│    │                  │                │
│                 │ • Analyze costs  │    │ • Evidence trace │                │
│                 │ • Find challenges│    │ • Logic check    │                │
│                 │ • Recommend      │    │ • Hallucination  │                │
│                 └────────┬─────────┘    └────────┬─────────┘                │
│                          │                       │                          │
│                          │◄──────────────────────┘                          │
│                          │  (Revise if validation fails)                    │
│                          ▼                                                  │
│                 ┌────────────────────────────────────────────┐              │
│                 │              FINAL REPORT                   │              │
│                 │  Validated, Evidence-Backed Recommendation  │              │
│                 └────────────────────────────────────────────┘              │
└──────────────────────────────────────────────────────────────────────────────┘
```

## Validator Agent's Unique Value

- **Evidence Traceability**: Ensures every claim traces back to research findings
- **Logical Consistency**: Detects internal contradictions and reasoning gaps
- **Quantitative Validation**: Cross-checks numerical claims against source data
- **Hallucination Detection**: Catches invented details not in research

## Example Use Case
**Query:** "Should we adopt GraphRAG for our search system?"

## 1. Environment Setup

In [ ]:
# Install required packages
%pip install -q langchain langchain-openai langchain-anthropic langchain-google-genai langgraph tavily-python pydantic

In [ ]:
import os
import json
import re
from typing import Annotated, List, TypedDict, Literal, Optional
from datetime import datetime
from enum import Enum

# LangChain imports
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage, BaseMessage, ToolMessage
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser, JsonOutputParser
from langchain_core.tools import tool
from pydantic import BaseModel, Field

# LangGraph imports
from langgraph.graph import StateGraph, END, START
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode

# Web search
from tavily import TavilyClient

print("✅ All imports successful!")

## 2. API Configuration

Configure your LLM provider and Tavily API key for web search.

In [ ]:
# Option 1: Set API keys directly (for testing - not recommended for production)
# os.environ["OPENAI_API_KEY"] = "your-openai-api-key"
# os.environ["TAVILY_API_KEY"] = "your-tavily-api-key"

# Option 2: Use getpass for secure input
from getpass import getpass

# Choose your LLM provider
LLM_PROVIDER = "openai"  # Options: "openai", "anthropic", "google"

if LLM_PROVIDER == "openai" and "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")
elif LLM_PROVIDER == "anthropic" and "ANTHROPIC_API_KEY" not in os.environ:
    os.environ["ANTHROPIC_API_KEY"] = getpass("Enter your Anthropic API key: ")
elif LLM_PROVIDER == "google" and "GOOGLE_API_KEY" not in os.environ:
    os.environ["GOOGLE_API_KEY"] = getpass("Enter your Google API key: ")

if "TAVILY_API_KEY" not in os.environ:
    os.environ["TAVILY_API_KEY"] = getpass("Enter your Tavily API key: ")

print(f"✅ Configured for {LLM_PROVIDER.upper()} with Tavily web search")

## 3. Initialize LLM

We'll use the same LLM for all agents but with different system prompts and temperatures.

In [ ]:
def get_llm(provider: str = "openai", temperature: float = 0.7):
    """Initialize the LLM based on the selected provider."""

    if provider == "openai":
        from langchain_openai import ChatOpenAI
        return ChatOpenAI(model="gpt-4o", temperature=temperature)

    elif provider == "anthropic":
        from langchain_anthropic import ChatAnthropic
        return ChatAnthropic(model="claude-sonnet-4-20250514", temperature=temperature)

    elif provider == "google":
        from langchain_google_genai import ChatGoogleGenerativeAI
        return ChatGoogleGenerativeAI(model="gemini-1.5-pro", temperature=temperature)

    else:
        raise ValueError(f"Unknown provider: {provider}")

# Initialize LLMs for different purposes
llm = get_llm(LLM_PROVIDER, temperature=0.7)
llm_structured = get_llm(LLM_PROVIDER, temperature=0.3)  # Lower temp for structured outputs
llm_validator = get_llm(LLM_PROVIDER, temperature=0.2)  # Very low temp for validation (more deterministic)

print(f"✅ LLMs initialized for {LLM_PROVIDER}")

## 4. Define Research Tools

The Researcher Agent has access to specialized search tools for different types of information.

In [ ]:
# Initialize Tavily client
tavily_client = TavilyClient(api_key=os.environ["TAVILY_API_KEY"])


@tool
def search_technical_docs(query: str) -> str:
    """Search for technical documentation, official guides, and API references.

    Args:
        query: The technical topic to search for documentation.

    Returns:
        Technical documentation and guides related to the query.
    """
    try:
        response = tavily_client.search(
            query=f"{query} technical documentation guide tutorial",
            search_depth="advanced",
            max_results=5,
            include_answer=True,
        )

        results = []
        if response.get("answer"):
            results.append(f"📚 **Summary:** {response['answer']}\n")

        results.append("**Technical Documentation Found:**\n")
        for i, r in enumerate(response.get("results", []), 1):
            results.append(f"{i}. **{r.get('title', 'N/A')}**\n   Source: {r.get('url', '')}\n   {r.get('content', '')[:300]}...\n")

        return "\n".join(results)
    except Exception as e:
        return f"Search error: {str(e)}"


@tool
def search_academic_papers(query: str) -> str:
    """Search for academic papers, research articles, and scientific publications.

    Args:
        query: The research topic to find academic papers about.

    Returns:
        Academic papers and research findings related to the query.
    """
    try:
        response = tavily_client.search(
            query=f"{query} research paper arxiv academic study",
            search_depth="advanced",
            max_results=5,
            include_answer=True,
        )

        results = []
        if response.get("answer"):
            results.append(f"📄 **Research Summary:** {response['answer']}\n")

        results.append("**Academic Papers & Research:**\n")
        for i, r in enumerate(response.get("results", []), 1):
            results.append(f"{i}. **{r.get('title', 'N/A')}**\n   Source: {r.get('url', '')}\n   {r.get('content', '')[:300]}...\n")

        return "\n".join(results)
    except Exception as e:
        return f"Search error: {str(e)}"


@tool
def search_benchmarks(query: str) -> str:
    """Search for performance benchmarks, comparisons, and metrics.

    Args:
        query: The technology/method to find benchmarks for.

    Returns:
        Performance benchmarks, comparisons, and metrics.
    """
    try:
        response = tavily_client.search(
            query=f"{query} benchmark performance comparison metrics evaluation",
            search_depth="advanced",
            max_results=5,
            include_answer=True,
        )

        results = []
        if response.get("answer"):
            results.append(f"📊 **Benchmark Summary:** {response['answer']}\n")

        results.append("**Benchmarks & Performance Data:**\n")
        for i, r in enumerate(response.get("results", []), 1):
            results.append(f"{i}. **{r.get('title', 'N/A')}**\n   Source: {r.get('url', '')}\n   {r.get('content', '')[:300]}...\n")

        return "\n".join(results)
    except Exception as e:
        return f"Search error: {str(e)}"


@tool
def search_implementation_examples(query: str) -> str:
    """Search for implementation examples, code samples, and practical guides.

    Args:
        query: The technology to find implementation examples for.

    Returns:
        Code examples, implementation guides, and practical tutorials.
    """
    try:
        response = tavily_client.search(
            query=f"{query} implementation example code github tutorial how to",
            search_depth="advanced",
            max_results=5,
            include_answer=True,
        )

        results = []
        if response.get("answer"):
            results.append(f"💻 **Implementation Summary:** {response['answer']}\n")

        results.append("**Implementation Examples:**\n")
        for i, r in enumerate(response.get("results", []), 1):
            results.append(f"{i}. **{r.get('title', 'N/A')}**\n   Source: {r.get('url', '')}\n   {r.get('content', '')[:300]}...\n")

        return "\n".join(results)
    except Exception as e:
        return f"Search error: {str(e)}"


@tool
def search_cost_analysis(query: str) -> str:
    """Search for cost analysis, pricing, and resource requirements.

    Args:
        query: The technology to analyze costs for.

    Returns:
        Cost analysis, pricing information, and resource requirements.
    """
    try:
        response = tavily_client.search(
            query=f"{query} cost pricing analysis resources requirements infrastructure",
            search_depth="advanced",
            max_results=5,
            include_answer=True,
        )

        results = []
        if response.get("answer"):
            results.append(f"💰 **Cost Summary:** {response['answer']}\n")

        results.append("**Cost & Resource Analysis:**\n")
        for i, r in enumerate(response.get("results", []), 1):
            results.append(f"{i}. **{r.get('title', 'N/A')}**\n   Source: {r.get('url', '')}\n   {r.get('content', '')[:300]}...\n")

        return "\n".join(results)
    except Exception as e:
        return f"Search error: {str(e)}"


# All research tools
research_tools = [
    search_technical_docs,
    search_academic_papers,
    search_benchmarks,
    search_implementation_examples,
    search_cost_analysis,
]

print("✅ Research tools defined:")
for t in research_tools:
    print(f"   🔧 {t.name}")

## 5. Define State Schema

The state tracks the conversation between agents, research findings, validation results, and analysis progress.

In [ ]:
class ValidationIssue(BaseModel):
    """A single validation issue found by the Validator Agent."""
    issue_type: str = Field(description="Type: credibility, contradiction, unsupported, gap, logic_error, hallucination, numerical_error")
    description: str = Field(description="Detailed description of the issue")
    severity: str = Field(description="Severity: critical, major, minor")
    affected_claim: str = Field(description="The specific claim or section affected")
    suggested_action: str = Field(description="What action should be taken to resolve this")


class ValidationResult(BaseModel):
    """Result from the Validator Agent."""
    is_valid: bool = Field(description="Whether the content passed validation")
    issues: List[ValidationIssue] = Field(default_factory=list, description="List of validation issues found")
    summary: str = Field(description="Summary of the validation results")
    confidence_score: float = Field(description="Confidence in the validation (0.0-1.0)")


class MultiAgentState(TypedDict):
    """State for the multi-agent research, validation, and analysis system.

    Attributes:
        messages: Full conversation history between agents
        user_query: Original user question
        research_findings: Accumulated research data
        research_validation_issues: Issues found in research validation (first pass)
        validated_research: Research after validation/refinement
        analysis_notes: Analyst's working notes
        analysis_validation_issues: Issues found in analysis validation (second pass)
        research_requests: Pending requests from analyst to researcher
        iteration_count: Number of research-analysis cycles
        research_refinement_count: Number of research refinement loops
        analysis_revision_count: Number of analysis revision loops
        current_agent: Which agent is currently active
        final_recommendation: The final analysis report
    """
    messages: Annotated[list, add_messages]
    user_query: str
    research_findings: List[str]
    research_validation_issues: List[str]
    validated_research: List[str]
    analysis_notes: List[str]
    analysis_validation_issues: List[str]
    research_requests: List[str]
    iteration_count: int
    research_refinement_count: int
    analysis_revision_count: int
    current_agent: str
    final_recommendation: str

print("✅ State schema defined")

## 6. Define Agent Prompts

Each agent has a specialized system prompt defining its role and capabilities.

In [ ]:
RESEARCHER_SYSTEM_PROMPT = """You are an expert Technical Researcher Agent. Your role is to gather
comprehensive information to support technical decision-making.

## Your Capabilities
You have access to specialized search tools:
- search_technical_docs: Find official documentation and guides
- search_academic_papers: Find research papers and studies
- search_benchmarks: Find performance comparisons and metrics
- search_implementation_examples: Find code examples and tutorials
- search_cost_analysis: Find pricing and resource requirements

## Your Task
Given a research request, systematically gather relevant information using your tools.
Focus on:
1. Technical documentation and official sources
2. Academic research and papers
3. Performance benchmarks and comparisons
4. Real-world implementation examples
5. Cost and resource considerations

## Guidelines
- Use multiple tools to get comprehensive coverage
- Prioritize recent and authoritative sources
- Note any conflicting information found
- Be thorough but focused on the specific request
- ALWAYS include source URLs for traceability
- Clearly distinguish between facts and interpretations

When you have gathered sufficient information, summarize your findings clearly with sources.
"""

VALIDATOR_RESEARCH_PROMPT = """You are an expert Validator Agent. Your role is to validate research
findings before they are used for analysis.

## Your Task (Research Validation - First Pass)
Validate the research findings for:

1. **Source Credibility**
   - Are sources authoritative (official docs, peer-reviewed papers, reputable tech blogs)?
   - Are sources recent enough to be relevant?
   - Are there any questionable or unreliable sources?

2. **Contradiction Detection**
   - Do any findings contradict each other?
   - Are there conflicting benchmarks or claims?
   - Note any areas of disagreement between sources

3. **Unsupported Assertions**
   - Are there claims without proper sources?
   - Are numerical claims backed by data?
   - Flag any speculation presented as fact

4. **Information Gaps**
   - What critical information is missing for the query?
   - Are there aspects of the question not covered?
   - What additional research would strengthen the findings?

## Output Format
Respond with a structured validation report in JSON format:
```json
{
    "is_valid": true/false,
    "issues": [
        {
            "issue_type": "credibility|contradiction|unsupported|gap",
            "description": "Detailed description",
            "severity": "critical|major|minor",
            "affected_claim": "The specific claim affected",
            "suggested_action": "What to do about it"
        }
    ],
    "summary": "Overall assessment",
    "confidence_score": 0.0-1.0
}
```

Set is_valid=true only if there are no critical issues and few major issues.
"""

VALIDATOR_ANALYSIS_PROMPT = """You are an expert Validator Agent. Your role is to validate
the analyst's conclusions before final delivery.

## Your Task (Analysis Validation - Second Pass)
Validate the analysis against the research findings for:

1. **Evidence Traceability**
   - Does every claim in the analysis trace back to research findings?
   - Are conclusions supported by the evidence presented?
   - Flag any claims that don't have supporting research

2. **Logical Consistency**
   - Are there any internal contradictions in the analysis?
   - Does the reasoning flow logically from premises to conclusions?
   - Are there any reasoning gaps or unsupported leaps?
   - Does the recommendation align with the stated constraints?

3. **Quantitative Validation**
   - Are numerical claims (e.g., "30% improvement") backed by source data?
   - Are cost estimates reasonable and well-sourced?
   - Are benchmark comparisons apples-to-apples?

4. **Hallucination Detection**
   - Does the analyst invent technical details not in the research?
   - Are features correctly attributed to the right tools/technologies?
   - Are there synthetic "facts" that don't appear in sources?

5. **Scope Adherence**
   - Does the analysis stay within what the research actually covered?
   - Are out-of-scope claims clearly identified as such?

## Output Format
Respond with a structured validation report in JSON format:
```json
{
    "is_valid": true/false,
    "issues": [
        {
            "issue_type": "evidence_missing|logic_error|numerical_error|hallucination|scope_violation",
            "description": "Detailed description",
            "severity": "critical|major|minor",
            "affected_claim": "The specific claim affected",
            "suggested_action": "What to do about it"
        }
    ],
    "summary": "Overall assessment",
    "confidence_score": 0.0-1.0
}
```

Set is_valid=true only if:
- All major claims trace to research evidence
- No logical inconsistencies
- No hallucinations detected
- Numerical claims are verified
"""

ANALYST_SYSTEM_PROMPT = """You are an expert Technical Analyst Agent. Your role is to analyze
research findings and provide actionable recommendations.

## Your Task
Analyze the VALIDATED research provided and create a comprehensive technical assessment.

## Analysis Framework
1. **Trade-off Analysis**: Compare options on key dimensions (performance, complexity, cost)
2. **Cost/Performance Implications**: Assess resource requirements and expected benefits
3. **Integration Challenges**: Identify potential issues with existing systems
4. **Risk Assessment**: Highlight uncertainties and potential problems
5. **Recommendation**: Provide clear guidance with supporting evidence

## CRITICAL GUIDELINES
- **ONLY use information from the provided research findings**
- **ALWAYS cite sources when making claims**
- **DO NOT invent facts, numbers, or technical details**
- **Clearly distinguish between research-backed facts and your interpretations**
- **If information is missing, note it as a limitation rather than making it up**

## Decision Points
After each analysis cycle, you must decide:
- **NEED_MORE_RESEARCH**: If you need specific additional information to complete your analysis
  - Specify exactly what information you need and why
- **READY_TO_CONCLUDE**: If you have enough information to make a well-supported recommendation

## Output Format
Structure your analysis with clear sections:
- Executive Summary
- Detailed Analysis (by category, with source citations)
- Key Findings (backed by research)
- Risks and Considerations
- Limitations (what the research didn't cover)
- Final Recommendation

Be specific, cite evidence from the research, and quantify claims when possible.
"""

print("✅ Agent prompts defined")

## 7. Define Agent Nodes

Each node represents a step in the multi-agent workflow.

In [ ]:
def initial_research(state: MultiAgentState) -> MultiAgentState:
    """Conduct initial comprehensive research on the user's query."""
    print("\n" + "="*60)
    print("🔬 RESEARCHER AGENT: Initial Research Phase")
    print("="*60)

    query = state["user_query"]

    # Create research prompt
    research_prompt = f"""Please conduct comprehensive research on the following question:

"{query}"

Use your available tools to gather:
1. Technical documentation and official information
2. Academic research and papers
3. Performance benchmarks and comparisons
4. Implementation examples
5. Cost considerations

Be thorough and systematic in your research. ALWAYS include source URLs for every finding."""

    # Bind tools to the LLM
    llm_with_tools = llm.bind_tools(research_tools)

    messages = [
        SystemMessage(content=RESEARCHER_SYSTEM_PROMPT),
        HumanMessage(content=research_prompt)
    ]

    # Let the researcher work with tools
    all_findings = []
    for i in range(5):  # Max 5 tool calls in initial research
        response = llm_with_tools.invoke(messages)
        messages.append(response)

        if not response.tool_calls:
            break

        # Execute tool calls
        for tool_call in response.tool_calls:
            tool_name = tool_call["name"]
            tool_args = tool_call["args"]
            tool_call_id = tool_call["id"]
            print(f"   🔧 Using tool: {tool_name}")

            # Find and execute the tool
            tool_fn = next((t for t in research_tools if t.name == tool_name), None)
            if tool_fn:
                result = tool_fn.invoke(tool_args)
                all_findings.append(f"[{tool_name}]: {result}")
                messages.append(ToolMessage(content=result, tool_call_id=tool_call_id))

    # Get final summary from researcher
    summary_prompt = "Please summarize all the research findings you've gathered in a structured format. Include source URLs for traceability."
    messages.append(HumanMessage(content=summary_prompt))
    summary = llm.invoke(messages)

    print(f"\n📋 Research Summary Generated")

    return {
        **state,
        "messages": [
            SystemMessage(content=RESEARCHER_SYSTEM_PROMPT),
            HumanMessage(content=f"Research Request: {query}"),
            AIMessage(content=summary.content)
        ],
        "research_findings": all_findings + [summary.content],
        "current_agent": "validate_research",
        "iteration_count": 1,
    }

In [ ]:
def validate_research(state: MultiAgentState) -> MultiAgentState:
    """Validator Agent: First pass - validate research findings."""
    print("\n" + "="*60)
    print("✅ VALIDATOR AGENT: Research Validation (First Pass)")
    print("="*60)

    findings_text = "\n\n---\n\n".join(state["research_findings"])

    validation_prompt = f"""Please validate the following research findings for the question:

**Original Question:** {state["user_query"]}

**Research Findings to Validate:**
{findings_text}

Perform your validation checks and return a structured JSON response."""

    messages = [
        SystemMessage(content=VALIDATOR_RESEARCH_PROMPT),
        HumanMessage(content=validation_prompt)
    ]

    response = llm_validator.invoke(messages)
    validation_result = response.content

    # Parse validation result
    try:
        # Extract JSON from response
        json_match = re.search(r'```json\s*([\s\S]*?)\s*```', validation_result)
        if json_match:
            result_json = json.loads(json_match.group(1))
        else:
            result_json = json.loads(validation_result)
        
        is_valid = result_json.get("is_valid", False)
        issues = result_json.get("issues", [])
        summary = result_json.get("summary", "")
        confidence = result_json.get("confidence_score", 0.5)
    except Exception:
        # If JSON parsing fails, do basic check
        is_valid = "is_valid\": true" in validation_result.lower() or "no critical issues" in validation_result.lower()
        issues = []
        summary = validation_result
        confidence = 0.5

    # Count critical and major issues
    critical_issues = [i for i in issues if i.get("severity") == "critical"]
    major_issues = [i for i in issues if i.get("severity") == "major"]
    
    print(f"\n📊 Validation Results:")
    print(f"   - Valid: {is_valid}")
    print(f"   - Critical Issues: {len(critical_issues)}")
    print(f"   - Major Issues: {len(major_issues)}")
    print(f"   - Confidence: {confidence:.2f}")
    
    if issues:
        print(f"\n   Issues Found:")
        for issue in issues[:5]:  # Show first 5 issues
            print(f"   ⚠️ [{issue.get('severity', 'unknown')}] {issue.get('issue_type', 'unknown')}: {issue.get('description', '')[:100]}...")

    # Determine if refinement is needed
    needs_refinement = not is_valid or len(critical_issues) > 0 or len(major_issues) > 2
    
    # Extract gap-related issues for research refinement
    gap_issues = [i for i in issues if i.get("issue_type") in ["gap", "unsupported"]]
    refinement_requests = [i.get("suggested_action", "") for i in gap_issues]

    return {
        **state,
        "research_validation_issues": [json.dumps(issues)],
        "validated_research": state["research_findings"] if is_valid else [],
        "current_agent": "refine_research" if needs_refinement and state["research_refinement_count"] < 2 else "analyst",
        "research_requests": refinement_requests,
    }

In [ ]:
def refine_research(state: MultiAgentState) -> MultiAgentState:
    """Researcher conducts additional research based on validator's feedback."""
    print("\n" + "="*60)
    print("🔬 RESEARCHER AGENT: Research Refinement Phase")
    print("="*60)

    # Get validation issues and refinement requests
    validation_issues = state["research_validation_issues"][-1] if state["research_validation_issues"] else "[]"
    refinement_requests = state["research_requests"]

    research_prompt = f"""The Validator Agent has identified issues with the research.

**Validation Issues:**
{validation_issues}

**Specific refinement requests:**
{chr(10).join(f'- {r}' for r in refinement_requests)}

Please conduct additional research to address these gaps and issues.
Focus on finding credible sources for unsupported claims and filling information gaps."""

    llm_with_tools = llm.bind_tools(research_tools)

    messages = [
        SystemMessage(content=RESEARCHER_SYSTEM_PROMPT),
        HumanMessage(content=research_prompt)
    ]

    new_findings = []
    for i in range(3):  # Max 3 additional tool calls for refinement
        response = llm_with_tools.invoke(messages)
        messages.append(response)

        if not response.tool_calls:
            break

        for tool_call in response.tool_calls:
            tool_name = tool_call["name"]
            tool_args = tool_call["args"]
            tool_call_id = tool_call["id"]
            print(f"   🔧 Refinement search: {tool_name}")

            tool_fn = next((t for t in research_tools if t.name == tool_name), None)
            if tool_fn:
                result = tool_fn.invoke(tool_args)
                new_findings.append(f"[Refinement - {tool_name}]: {result}")
                messages.append(ToolMessage(content=result, tool_call_id=tool_call_id))

    # Summarize new findings
    messages.append(HumanMessage(content="Summarize the additional findings with source URLs."))
    summary = llm.invoke(messages)
    new_findings.append(summary.content)

    print(f"\n📋 Research refinement complete")

    return {
        **state,
        "messages": state["messages"] + [AIMessage(content=summary.content)],
        "research_findings": state["research_findings"] + new_findings,
        "current_agent": "validate_research",
        "research_refinement_count": state["research_refinement_count"] + 1,
    }

In [ ]:
def analyze_research(state: MultiAgentState) -> MultiAgentState:
    """Analyst reviews validated research and creates analysis."""
    print("\n" + "="*60)
    print("📊 ANALYST AGENT: Analysis Phase")
    print("="*60)

    # Use validated research or all research if validation passed
    findings = state["validated_research"] if state["validated_research"] else state["research_findings"]
    findings_text = "\n\n---\n\n".join(findings)

    # Include any previous validation feedback
    analysis_feedback = ""
    if state["analysis_validation_issues"]:
        analysis_feedback = f"""\n\n**Previous Analysis Issues to Address:**
{state["analysis_validation_issues"][-1]}

Please revise your analysis to address these issues."""

    analysis_prompt = f"""Please analyze the following VALIDATED research findings for the question:

**Original Question:** {state["user_query"]}

**Validated Research Findings:**
{findings_text}
{analysis_feedback}

**Instructions:**
1. Analyze the trade-offs and implications
2. ONLY use information from the research above - DO NOT invent facts
3. CITE SOURCES for every claim you make
4. Note any limitations or gaps in the available research
5. Either:
   - Request specific additional research by saying "NEED_MORE_RESEARCH:" followed by what you need
   - Or provide your complete analysis if you have enough information

Remember: Your analysis will be validated. Unsupported claims will be flagged."""

    messages = [
        SystemMessage(content=ANALYST_SYSTEM_PROMPT),
        HumanMessage(content=analysis_prompt)
    ]

    response = llm.invoke(messages)
    analysis = response.content

    print(f"\n📝 Analysis complete")

    # Check if analyst needs more research
    needs_more = "NEED_MORE_RESEARCH" in analysis.upper()

    return {
        **state,
        "messages": state["messages"] + [
            HumanMessage(content="Please analyze the research and provide your assessment."),
            AIMessage(content=analysis)
        ],
        "analysis_notes": state["analysis_notes"] + [analysis],
        "current_agent": "additional_research" if needs_more else "validate_analysis",
        "research_requests": [analysis] if needs_more else [],
    }

In [ ]:
def additional_research(state: MultiAgentState) -> MultiAgentState:
    """Researcher conducts additional research based on analyst's request."""
    print("\n" + "="*60)
    print("🔬 RESEARCHER AGENT: Additional Research Phase")
    print("="*60)

    # Extract what the analyst needs
    last_analysis = state["analysis_notes"][-1] if state["analysis_notes"] else ""

    research_prompt = f"""The analyst has requested additional research.

Their analysis so far:
{last_analysis}

Please search for the specific information they need to complete their analysis.
Focus on filling the gaps they've identified. Include source URLs."""

    llm_with_tools = llm.bind_tools(research_tools)

    messages = [
        SystemMessage(content=RESEARCHER_SYSTEM_PROMPT),
        HumanMessage(content=research_prompt)
    ]

    new_findings = []
    for i in range(3):  # Max 3 additional tool calls
        response = llm_with_tools.invoke(messages)
        messages.append(response)

        if not response.tool_calls:
            break

        for tool_call in response.tool_calls:
            tool_name = tool_call["name"]
            tool_args = tool_call["args"]
            tool_call_id = tool_call["id"]
            print(f"   🔧 Additional search: {tool_name}")

            tool_fn = next((t for t in research_tools if t.name == tool_name), None)
            if tool_fn:
                result = tool_fn.invoke(tool_args)
                new_findings.append(f"[Additional - {tool_name}]: {result}")
                messages.append(ToolMessage(content=result, tool_call_id=tool_call_id))

    # Summarize new findings
    messages.append(HumanMessage(content="Summarize the additional findings."))
    summary = llm.invoke(messages)
    new_findings.append(summary.content)

    print(f"\n📋 Additional research complete")

    return {
        **state,
        "messages": state["messages"] + [AIMessage(content=summary.content)],
        "research_findings": state["research_findings"] + new_findings,
        "validated_research": state["validated_research"] + new_findings,
        "current_agent": "analyst",
        "iteration_count": state["iteration_count"] + 1,
    }

In [ ]:
def validate_analysis(state: MultiAgentState) -> MultiAgentState:
    """Validator Agent: Second pass - validate analysis against research."""
    print("\n" + "="*60)
    print("✅ VALIDATOR AGENT: Analysis Validation (Second Pass)")
    print("="*60)

    # Get the research findings and analysis
    research_text = "\n\n---\n\n".join(state["research_findings"])
    analysis_text = state["analysis_notes"][-1] if state["analysis_notes"] else ""

    validation_prompt = f"""Please validate the analyst's conclusions against the research findings.

**Original Question:** {state["user_query"]}

**Research Findings (Source of Truth):**
{research_text}

**Analysis to Validate:**
{analysis_text}

Verify that:
1. Every claim traces back to the research findings
2. The logic is consistent and conclusions follow from evidence
3. Numerical claims match the source data
4. No invented facts or hallucinations
5. Recommendation aligns with stated constraints

Return your validation in structured JSON format."""

    messages = [
        SystemMessage(content=VALIDATOR_ANALYSIS_PROMPT),
        HumanMessage(content=validation_prompt)
    ]

    response = llm_validator.invoke(messages)
    validation_result = response.content

    # Parse validation result
    try:
        json_match = re.search(r'```json\s*([\s\S]*?)\s*```', validation_result)
        if json_match:
            result_json = json.loads(json_match.group(1))
        else:
            result_json = json.loads(validation_result)
        
        is_valid = result_json.get("is_valid", False)
        issues = result_json.get("issues", [])
        summary = result_json.get("summary", "")
        confidence = result_json.get("confidence_score", 0.5)
    except Exception:
        is_valid = "is_valid\": true" in validation_result.lower() or "no critical issues" in validation_result.lower()
        issues = []
        summary = validation_result
        confidence = 0.5

    # Count critical issues
    critical_issues = [i for i in issues if i.get("severity") == "critical"]
    major_issues = [i for i in issues if i.get("severity") == "major"]
    
    print(f"\n📊 Analysis Validation Results:")
    print(f"   - Valid: {is_valid}")
    print(f"   - Critical Issues: {len(critical_issues)}")
    print(f"   - Major Issues: {len(major_issues)}")
    print(f"   - Confidence: {confidence:.2f}")
    
    if issues:
        print(f"\n   Issues Found:")
        for issue in issues[:5]:
            print(f"   ⚠️ [{issue.get('severity', 'unknown')}] {issue.get('issue_type', 'unknown')}: {issue.get('description', '')[:100]}...")

    # Determine if revision is needed
    needs_revision = not is_valid or len(critical_issues) > 0
    can_revise = state["analysis_revision_count"] < 2  # Max 2 revision loops

    if needs_revision and not can_revise:
        print("\n⚠️ Max revision attempts reached - escalating to human review")

    # Determine next agent
    if needs_revision and can_revise:
        next_agent = "revise_analysis"
    elif needs_revision and not can_revise:
        next_agent = "human_review"  # Escalation path
    else:
        next_agent = "final_report"

    return {
        **state,
        "analysis_validation_issues": state["analysis_validation_issues"] + [json.dumps(issues)],
        "current_agent": next_agent,
    }

In [ ]:
def revise_analysis(state: MultiAgentState) -> MultiAgentState:
    """Analyst revises their analysis based on validator feedback."""
    print("\n" + "="*60)
    print("📊 ANALYST AGENT: Analysis Revision Phase")
    print("="*60)

    # Get validation issues
    validation_issues = state["analysis_validation_issues"][-1] if state["analysis_validation_issues"] else "[]"
    previous_analysis = state["analysis_notes"][-1] if state["analysis_notes"] else ""
    research_text = "\n\n---\n\n".join(state["research_findings"])

    revision_prompt = f"""The Validator Agent has identified issues with your analysis that need to be addressed.

**Original Question:** {state["user_query"]}

**Your Previous Analysis:**
{previous_analysis}

**Validation Issues to Address:**
{validation_issues}

**Research Findings (Source of Truth):**
{research_text}

**Instructions:**
1. Address each validation issue identified
2. Remove or correct any unsupported claims
3. Add proper source citations where missing
4. Fix any logical inconsistencies
5. Ensure all numerical claims match the research data

Provide your REVISED analysis addressing all the validation feedback."""

    messages = [
        SystemMessage(content=ANALYST_SYSTEM_PROMPT),
        HumanMessage(content=revision_prompt)
    ]

    response = llm.invoke(messages)
    revised_analysis = response.content

    print(f"\n📝 Analysis revision complete")

    return {
        **state,
        "messages": state["messages"] + [
            HumanMessage(content="Please revise your analysis based on the validation feedback."),
            AIMessage(content=revised_analysis)
        ],
        "analysis_notes": state["analysis_notes"] + [revised_analysis],
        "current_agent": "validate_analysis",
        "analysis_revision_count": state["analysis_revision_count"] + 1,
    }

In [ ]:
def human_review(state: MultiAgentState) -> MultiAgentState:
    """Escalation path when validation keeps failing - flag for human review."""
    print("\n" + "="*60)
    print("👤 ESCALATION: Human Review Required")
    print("="*60)

    escalation_note = f"""
⚠️ HUMAN REVIEW REQUIRED ⚠️

The automated validation process was unable to fully verify the analysis after multiple attempts.

Outstanding validation issues:
{state["analysis_validation_issues"][-1] if state["analysis_validation_issues"] else "Unknown"}

Please review the analysis manually before proceeding with the recommendation.
"""

    print(escalation_note)

    return {
        **state,
        "current_agent": "final_report",
        "messages": state["messages"] + [AIMessage(content=escalation_note)],
    }

In [ ]:
def generate_final_report(state: MultiAgentState) -> MultiAgentState:
    """Generate the final comprehensive recommendation report."""
    print("\n" + "="*60)
    print("📄 GENERATING FINAL VALIDATED REPORT")
    print("="*60)

    all_research = "\n\n".join(state["research_findings"])
    all_analysis = "\n\n".join(state["analysis_notes"])
    
    # Check if there were unresolved validation issues
    has_unresolved_issues = state["analysis_revision_count"] >= 2 and state["analysis_validation_issues"]

    final_prompt = f"""Based on all the VALIDATED research and analysis conducted, create a comprehensive
final recommendation report for the question:

**Question:** {state["user_query"]}

**Validated Research Findings:**
{all_research}

**Validated Analysis:**
{all_analysis}

**Validation Status:**
- Research validation passes: {state["research_refinement_count"]} refinements needed
- Analysis validation passes: {state["analysis_revision_count"]} revisions needed
- Unresolved issues: {"Yes - marked for human review" if has_unresolved_issues else "No"}

Create a professional report with:
1. **Executive Summary** - Key takeaway in 2-3 sentences
2. **Evidence Summary** - Key findings with source citations
3. **Technology Comparison** - Based on validated research
4. **Performance Analysis** - Verified benchmarks and metrics
5. **Cost Analysis** - Resource and infrastructure requirements
6. **Integration Considerations** - Challenges with existing systems
7. **Risk Assessment** - Potential issues and mitigations
8. **Validation Notes** - Confidence level and any caveats
9. **Final Recommendation** - Clear guidance with confidence level
10. **Next Steps** - Actionable items if adopting

Be specific, cite evidence, and clearly note any remaining uncertainties.
{'⚠️ Include a note about unresolved validation issues requiring human review.' if has_unresolved_issues else ''}"""

    messages = [
        SystemMessage(content="You are an expert technical writer creating a final, validated recommendation report. All claims must be traced to research evidence."),
        HumanMessage(content=final_prompt)
    ]

    response = llm.invoke(messages)

    return {
        **state,
        "final_recommendation": response.content,
        "current_agent": "complete",
    }


print("✅ Agent nodes defined")

## 8. Define Routing Logic

The router directs flow between agents based on validation results and analysis state.

In [ ]:
def route_after_research_validation(state: MultiAgentState) -> Literal["refine_research", "analyze"]:
    """Route based on research validation results."""
    if state["current_agent"] == "refine_research":
        print("🔄 Validator requested research refinement")
        return "refine_research"
    else:
        print("✅ Research validation passed - proceeding to analysis")
        return "analyze"


def route_after_analysis(state: MultiAgentState) -> Literal["additional_research", "validate_analysis"]:
    """Route based on whether analyst needs more research."""
    if state["iteration_count"] >= 3:
        print("⏹️ Max iterations reached, proceeding to validation")
        return "validate_analysis"

    if state["current_agent"] == "additional_research":
        print("🔄 Analyst requested additional research")
        return "additional_research"
    else:
        print("✅ Analysis complete - proceeding to validation")
        return "validate_analysis"


def route_after_analysis_validation(state: MultiAgentState) -> Literal["revise_analysis", "human_review", "final_report"]:
    """Route based on analysis validation results."""
    if state["current_agent"] == "revise_analysis":
        print("🔄 Validator requested analysis revision")
        return "revise_analysis"
    elif state["current_agent"] == "human_review":
        print("⚠️ Escalating to human review")
        return "human_review"
    else:
        print("✅ Analysis validation passed - generating final report")
        return "final_report"


print("✅ Routing logic defined")

## 9. Build the Multi-Agent Graph

Assemble the complete workflow with LangGraph.

In [ ]:
def build_validated_multi_agent_system():
    """Build the Researcher + Validator + Analyst multi-agent graph."""

    workflow = StateGraph(MultiAgentState)

    # Add nodes
    workflow.add_node("initial_research", initial_research)
    workflow.add_node("validate_research", validate_research)
    workflow.add_node("refine_research", refine_research)
    workflow.add_node("analyze", analyze_research)
    workflow.add_node("additional_research", additional_research)
    workflow.add_node("validate_analysis", validate_analysis)
    workflow.add_node("revise_analysis", revise_analysis)
    workflow.add_node("human_review", human_review)
    workflow.add_node("final_report", generate_final_report)

    # Define edges
    workflow.add_edge(START, "initial_research")
    workflow.add_edge("initial_research", "validate_research")
    
    # After research validation: either refine or proceed to analysis
    workflow.add_conditional_edges(
        "validate_research",
        route_after_research_validation,
        {
            "refine_research": "refine_research",
            "analyze": "analyze",
        }
    )
    
    # After research refinement, go back to validation
    workflow.add_edge("refine_research", "validate_research")
    
    # After analysis: either get more research or validate
    workflow.add_conditional_edges(
        "analyze",
        route_after_analysis,
        {
            "additional_research": "additional_research",
            "validate_analysis": "validate_analysis",
        }
    )
    
    # After additional research, go back to analyst
    workflow.add_edge("additional_research", "analyze")
    
    # After analysis validation: revise, escalate, or generate report
    workflow.add_conditional_edges(
        "validate_analysis",
        route_after_analysis_validation,
        {
            "revise_analysis": "revise_analysis",
            "human_review": "human_review",
            "final_report": "final_report",
        }
    )
    
    # After analysis revision, go back to validation
    workflow.add_edge("revise_analysis", "validate_analysis")
    
    # After human review, proceed to final report
    workflow.add_edge("human_review", "final_report")

    # Final report ends the workflow
    workflow.add_edge("final_report", END)

    return workflow.compile()


# Build the system
validated_multi_agent_system = build_validated_multi_agent_system()

print("✅ Validated Multi-Agent System built successfully!")
print("\nWorkflow:")
print("START → Initial Research → Validate Research ⟷ Refine Research")
print("                              ↓")
print("                          Analyze ⟷ Additional Research")
print("                              ↓")
print("                    Validate Analysis ⟷ Revise Analysis")
print("                              ↓")
print("                    Human Review (if needed)")
print("                              ↓")
print("                    Final Report → END")

In [ ]:
# Visualize the graph (optional)
try:
    from IPython.display import Image, display
    display(Image(validated_multi_agent_system.get_graph().draw_mermaid_png()))
except Exception as e:
    print("Graph visualization not available.")
    print("\nWorkflow structure:")
    print("""
    ┌─────────────────┐
    │      START      │
    └────────┬────────┘
             │
             ▼
    ┌─────────────────┐
    │ Initial Research│
    └────────┬────────┘
             │
             ▼
    ┌─────────────────┐     ┌─────────────────┐
    │   Validate      │────►│ Refine          │
    │   Research      │◄────│ Research        │
    └────────┬────────┘     └─────────────────┘
             │
             ▼
    ┌─────────────────┐     ┌─────────────────┐
    │    Analyze      │────►│ Additional      │
    │   (Analyst)     │◄────│ Research        │
    └────────┬────────┘     └─────────────────┘
             │
             ▼
    ┌─────────────────┐     ┌─────────────────┐
    │   Validate      │────►│ Revise          │
    │   Analysis      │◄────│ Analysis        │
    └────────┬────────┘     └─────────────────┘
             │
             ▼ (if max revisions)
    ┌─────────────────┐
    │  Human Review   │
    └────────┬────────┘
             │
             ▼
    ┌─────────────────┐
    │  Final Report   │
    └────────┬────────┘
             │
             ▼
    ┌─────────────────┐
    │      END        │
    └─────────────────┘
    """)

## 10. Run the Multi-Agent System

Let's test with the GraphRAG example query!

In [ ]:
def run_validated_analysis(query: str) -> str:
    """Run the multi-agent research, validation, and analysis system.

    Args:
        query: The technical question to research and analyze.

    Returns:
        The final validated recommendation report.
    """
    print("="*70)
    print("🤖 MULTI-AGENT RESEARCH, VALIDATION & ANALYSIS SYSTEM")
    print("="*70)
    print(f"\n📌 Query: {query}\n")
    print("="*70)

    # Initialize state
    initial_state = {
        "messages": [],
        "user_query": query,
        "research_findings": [],
        "research_validation_issues": [],
        "validated_research": [],
        "analysis_notes": [],
        "analysis_validation_issues": [],
        "research_requests": [],
        "iteration_count": 0,
        "research_refinement_count": 0,
        "analysis_revision_count": 0,
        "current_agent": "researcher",
        "final_recommendation": "",
    }

    # Run the system
    final_state = validated_multi_agent_system.invoke(initial_state)

    # Display final report
    print("\n" + "="*70)
    print("📊 FINAL VALIDATED RECOMMENDATION REPORT")
    print("="*70)
    print(final_state["final_recommendation"])
    print("\n" + "="*70)
    print(f"📈 Statistics:")
    print(f"   - Total research iterations: {final_state['iteration_count']}")
    print(f"   - Research refinements: {final_state['research_refinement_count']}")
    print(f"   - Analysis revisions: {final_state['analysis_revision_count']}")
    print("="*70)

    return final_state["final_recommendation"]

In [ ]:
# Primary Example: GraphRAG Adoption Decision
graphrag_query = """Should we adopt GraphRAG for our search system?

Context:
- We currently have a traditional RAG system using vector embeddings
- Our documents contain complex relationships between entities
- We need to handle multi-hop reasoning queries
- Budget and infrastructure constraints are a consideration

Please provide a comprehensive analysis and recommendation."""

report = run_validated_analysis(graphrag_query)

## 11. Additional Example Queries

Try other technical decision queries:

In [ ]:
# Example 2: LLM Framework Selection
framework_query = """Should we use LangChain or LlamaIndex for our enterprise RAG application?

Context:
- Building a document Q&A system for internal knowledge base
- Need to support multiple document types (PDF, Word, HTML)
- Require integration with Azure OpenAI
- Team has Python experience but new to LLM frameworks

Please compare and recommend."""

# Uncomment to run:
# report = run_validated_analysis(framework_query)

In [ ]:
# Example 3: Vector Database Selection
vectordb_query = """Which vector database should we choose: Pinecone, Weaviate, or Milvus?

Context:
- Expected to store 10 million+ embeddings
- Need low-latency queries (<100ms p99)
- Want to minimize operational overhead
- Budget is flexible but cost-efficiency matters
- Running on AWS infrastructure

Please analyze and recommend."""

# Uncomment to run:
# report = run_validated_analysis(vectordb_query)

In [ ]:
# Try your own technical decision query!
custom_query = """
Your question here...

Context:
- Add relevant context
- Include constraints
- Mention current setup
"""

# Uncomment to run:
# report = run_validated_analysis(custom_query)

## 12. Summary

This notebook demonstrates a **Multi-Agent Research, Validation & Analysis System** with three specialized agents working together iteratively.

### Agent Roles

| Agent | Role | Capabilities |
|-------|------|--------------|
| 🔬 **Researcher** | Information Gathering | Search docs, papers, benchmarks, examples, costs |
| ✅ **Validator** | Quality Assurance | Source credibility, contradictions, evidence tracing, hallucination detection |
| 📊 **Analyst** | Critical Analysis | Compare trade-offs, assess costs, identify challenges, recommend |

### Workflow Features

1. **Initial Research Phase**
   - Researcher conducts comprehensive information gathering
   - Uses 5 specialized search tools
   - Produces structured findings with sources

2. **Research Validation (First Pass)**
   - Validator checks source credibility
   - Detects contradictions and unsupported claims
   - Identifies information gaps
   - Can request research refinement (max 2 loops)

3. **Analysis Phase**
   - Analyst reviews validated research
   - Creates evidence-backed analysis with citations
   - Can request additional research if needed

4. **Analysis Validation (Second Pass)**
   - Validator verifies claims trace to research
   - Checks logical consistency
   - Validates numerical claims
   - Detects hallucinations
   - Can request revisions (max 2 loops)

5. **Human Review Escalation**
   - If validation keeps failing after max attempts
   - Flags report for human review

6. **Final Report**
   - Comprehensive validated recommendation
   - Includes confidence level and caveats
   - Notes any unresolved validation issues

### Key Benefits

- ✅ **Separation of concerns**: Research, validation, and analysis are distinct
- ✅ **Quality assurance**: Two-pass validation catches errors and hallucinations
- ✅ **Evidence traceability**: All claims linked to sources
- ✅ **Iterative refinement**: Gaps and issues are addressed
- ✅ **Escalation path**: Human review when automation fails
- ✅ **Transparent**: Full history of validation and revision steps

In [ ]:
print("\n🎉 Multi-Agent Research, Validation & Analysis System Complete!")
print("\nThis system demonstrates:")
print("  • Researcher Agent: Systematic information gathering with specialized tools")
print("  • Validator Agent: Two-pass quality assurance (research + analysis)")
print("  • Analyst Agent: Evidence-backed analysis and recommendation")
print("  • Iterative Loops: Research refinement and analysis revision")
print("  • Escalation Path: Human review when validation fails")
print("  • Final Report: Validated recommendation with confidence level")
print("\nModify the prompts or add new tools to customize for your use case!")